# L18: 多 Agent 协作

> 让多个专业 Agent 协同工作完成复杂任务

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.agents.multi_agent import MultiAgentOrchestrator
print("✓ 模块导入成功")

## 18.1 为什么需要多 Agent？

### 单 Agent 的局限性

```
单 Agent:
  - 知识广度有限
  - 难以同时专注多个任务
  - 没有专业细分
  - 无法并行处理

多 Agent:
  + 每个 Agent 专注一个领域
  + 可以并行工作
  + 互相协作解决问题
  + 可以互相审查和改进
```

### 多 Agent 协作模式

| 模式 | 说明 | 示例 |
|------|------|------|
| **并行** | 多 Agent 同时工作 | 多个研究员同时查资料 |
| **顺序** | 一个接一个执行 | 写作 → 审稿 → 编辑 |
| **层级** | 上层分配下层执行 | 经理 → 专员 |
| **辩论** | 互相讨论改进 | 两个方案辩论选最优 |
| **投票** | 多数决策 | 三个 Agent 投票 |
| **路由** | 根据类型分发 | 不同问题给不同专家 |

## 18.2 查看项目中的多 Agent 实现

In [ ]:
import inspect
from backend.agents.multi_agent import MultiAgentOrchestrator

print("=== MultiAgentOrchestrator 源码（部分）===\n")
source = inspect.getsource(MultiAgentOrchestrator)
print(source[:2000] + "\n... (源码已截断)")

## 18.3 专业 Agent 定义

### 典型角色分工

In [ ]:
from abc import ABC, abstractmethod

class BaseAgent(ABC):
    """
    Agent 基类
    """
    
    def __init__(self, name: str, role: str, goal: str):
        self.name = name
        self.role = role
        self.goal = goal
    
    @abstractmethod
    async def execute(self, task: str, context: Dict = None) -> str:
        """
        执行任务
        """
        pass
    
    def get_system_prompt(self) -> str:
        """
        获取系统提示
        """
        return f"""你是 {self.name}，你的角色是 {self.role}。

你的目标: {self.goal}

请根据你的专业能力完成任务。
"""

# 定义专业 Agent
class ResearcherAgent(BaseAgent):
    """研究专员 - 负责信息收集"""
    
    def __init__(self):
        super().__init__(
            name="研究员",
            role="信息收集和初步分析",
            goal="收集准确、全面的信息"
        )
    
    async def execute(self, task: str, context: Dict = None) -> str:
        # 模拟研究工作
        return f"[研究员] 已收集关于 '{task}' 的资料：包含3个可靠来源，信息覆盖面完整。"

class WriterAgent(BaseAgent):
    """写作专员 - 负责内容创作"""
    
    def __init__(self):
        super().__init__(
            name="写作者",
            role="内容创作和文案编写",
            goal="创作清晰、有吸引力的内容"
        )
    
    async def execute(self, task: str, context: Dict = None) -> str:
        research = context.get("research", "") if context else ""
        return f"[写作者] 基于研究资料，已完成 '{task}' 的内容创作。"

class ReviewerAgent(BaseAgent):
    """审稿专员 - 负责质量检查"""
    
    def __init__(self):
        super().__init__(
            name="审稿人",
            role="内容审查和质量控制",
            goal="确保内容准确、专业、易懂"
        )
    
    async def execute(self, task: str, context: Dict = None) -> str:
        content = context.get("content", "") if context else ""
        return f"[审稿人] 审查完成，发现 2 个可改进点，整体质量良好。"

# 创建 Agent 实例
researcher = ResearcherAgent()
writer = WriterAgent()
reviewer = ReviewerAgent()

print("=== 专业 Agent 已创建 ===\n")
for agent in [researcher, writer, reviewer]:
    print(f"📋 {agent.name}")
    print(f"   角色: {agent.role}")
    print(f"   目标: {agent.goal}")
    print()

## 18.4 顺序协作模式

### 也就是「流水线」模式

In [ ]:
class SequentialOrchestrator:
    """
    顺序执行编排器
    """
    
    def __init__(self, agents: List[BaseAgent]):
        self.agents = agents
    
    async def execute(self, task: str) -> Dict[str, Any]:
        """
        按顺序执行所有 Agent
        """
        context = {}
        results = []
        
        for i, agent in enumerate(self.agents):
            print(f"\n🔄 步骤 {i+1}: {agent.name} 执行中...")
            
            result = await agent.execute(task, context)
            results.append({
                "agent": agent.name,
                "result": result
            })
            
            # 将结果传递给下一个 Agent
            context[f"step_{i+1}_result"] = result
            context[f"last_agent"] = agent.name
            
            print(f"✓ {agent.name} 完成")
        
        return {
            "task": task,
            "results": results,
            "final_context": context
        }

# 测试顺序协作
async def test_sequential():
    print("=== 顺序协作演示 ===")
    
    orchestrator = SequentialOrchestrator([researcher, writer, reviewer])
    
    result = await orchestrator.execute("写一篇关于 Python 的文章")
    
    print("\n=== 执行结果 ===")
    for r in result["results"]:
        print(f"{r['agent']}: {r['result']}")

await test_sequential()

## 18.5 并行协作模式

### 多个 Agent 同时工作

In [ ]:
class ParallelOrchestrator:
    """
    并行执行编排器
    """
    
    def __init__(self, agents: List[BaseAgent]):
        self.agents = agents
    
    async def execute(self, task: str) -> Dict[str, Any]:
        """
        并行执行所有 Agent
        """
        print(f"🚀 并行启动 {len(self.agents)} 个 Agent...\n")
        
        # 创建并行任务
        tasks = [
            self._run_agent(agent, task)
            for agent in self.agents
        ]
        
        # 并行执行
        results = await asyncio.gather(*tasks)
        
        print("\n✓ 所有 Agent 完成")
        
        return {
            "task": task,
            "results": results
        }
    
    async def _run_agent(self, agent: BaseAgent, task: str) -> Dict[str, str]:
        """
        运行单个 Agent
        """
        result = await agent.execute(task)
        return {
            "agent": agent.name,
            "result": result
        }

# 创建多个研究员进行并行研究
async def test_parallel():
    print("=== 并行协作演示 ===\n")
    
    # 创建多个不同专长的研究员
    researchers = [
        ResearcherAgent(),  # 重命名以模拟不同专家
        ResearcherAgent(),
        ResearcherAgent(),
    ]
    
    # 给每个研究员不同的专长
    experts = ["技术", "市场", "竞品"]
    for agent, expert in zip(researchers, experts):
        agent.name = f"{expert}专家"
        
    orchestrator = ParallelOrchestrator(researchers)
    
    result = await orchestrator.execute("调研 AI 编程助手市场")
    
    print("\n=== 并行结果 ===")
    for r in result["results"]:
        print(f"{r['agent']}: {r['result']}")

await test_parallel()

## 18.6 路由模式

### 根据任务类型分发给不同的 Agent

In [ ]:
class RouterAgent:
    """
    路由 Agent - 决定将任务分发给谁
    """
    
    def __init__(self, agents: Dict[str, BaseAgent]):
        self.agents = agents
        self.routes = {
            "研究": ["researcher", "研究员", "调研", "分析"],
            "写作": ["writer", "写作者", "写作", "创作", "文章"],
            "审查": ["reviewer", "审稿人", "审查", "检查", "校对"],
            "计算": ["calculator", "计算器", "计算", "数学"],
        }
    
    def route(self, task: str) -> BaseAgent:
        """
        根据任务决定路由到哪个 Agent
        """
        for category, keywords in self.routes.items():
            if any(kw in task for kw in keywords):
                # 找到对应类别的 Agent
                for agent in self.agents.values():
                    if category.lower() in agent.name.lower():
                        return agent
        
        # 默认返回第一个
        return list(self.agents.values())[0]
    
    async def execute(self, task: str) -> str:
        """
        路由并执行任务
        """
        agent = self.route(task)
        print(f"🔀 路由: '{task}' → {agent.name}")
        
        result = await agent.execute(task)
        return result

# 测试路由
async def test_routing():
    print("=== 路由模式演示 ===\n")
    
    agents = {
        "researcher": researcher,
        "writer": writer,
        "reviewer": reviewer,
    }
    
    router = RouterAgent(agents)
    
    tasks = [
        "帮我调研一下市场",
        "写一篇产品介绍",
        "审查这篇文档",
    ]
    
    for task in tasks:
        result = await router.execute(task)
        print(f"结果: {result}\n")

await test_routing()

## 18.7 辩论模式

### 两个 Agent 产生不同方案，然后讨论

In [ ]:
class DebateOrchestrator:
    """
    辩论式编排器
    """
    
    def __init__(self, agent1: BaseAgent, agent2: BaseAgent, judge: BaseAgent):
        self.agent1 = agent1  # 提出方案 A
        self.agent2 = agent2  # 提出方案 B
        self.judge = judge   # 判断哪个更好
    
    async def execute(self, task: str) -> Dict[str, Any]:
        """
        执行辩论流程
        """
        print(f"🎯 任务: {task}\n")
        
        # 1. 双方提出方案
        print("📝 方案生成阶段...")
        proposal1 = await self.agent1.execute(f"为任务提出方案: {task}")
        proposal2 = await self.agent2.execute(f"为任务提出方案: {task}")
        
        print(f"\n方案 A ({self.agent1.name}): {proposal1[:50]}...")
        print(f"方案 B ({self.agent2.name}): {proposal2[:50]}...")
        
        # 2. 互相评论
        print("\n💬 辩论阶段...")
        comment1 = await self.agent1.execute(
            f"评论对方方案: {proposal2}。你的方案是: {proposal1}"
        )
        comment2 = await self.agent2.execute(
            f"评论对方方案: {proposal1}。你的方案是: {proposal2}"
        )
        
        print(f"{self.agent1.name} 的评论: {comment1[:50]}...")
        print(f"{self.agent2.name} 的评论: {comment2[:50]}...")
        
        # 3. 裁决
        print("\n⚖️ 裁决阶段...")
        decision = await self.judge.execute(
            f"根据以下信息裁决:\n方案A: {proposal1}\n评论A: {comment1}\n"
            f"方案B: {proposal2}\n评论B: {comment2}"
        )
        
        return {
            "task": task,
            "proposal_a": proposal1,
            "proposal_b": proposal2,
            "decision": decision
        }

# 模拟演示
async def demo_debate():
    print("=== 辩论模式演示 ===\n")
    print("(模拟运行，不实际调用 LLM)\n")
    
    # 模拟流程
    print("1️⃣ 方案生成:")
    print("   方案 A: 采用传统方法，稳定可靠")
    print("   方案 B: 采用创新方法，效率更高")
    print("\n2️⃣ 辩论:")
    print("   方案 A 认为风险控制更重要")
    print("   方案 B 认为创新能带来竞争优势")
    print("\n3️⃣ 裁决:")
    print("   综合考虑，推荐采用渐进式创新方案")

await demo_debate()

## 18.8 常见面试问题

**Q: 多 Agent 协作的优势是什么？**
- A:
  1. **专业分工**：每个 Agent 专注一个领域
  2. **并行处理**：多个任务同时进行
  3. **质量提升**：互相审查和改进
  4. **容错能力**：一个失败不影响其他
  5. **可扩展性**：容易添加新的专业 Agent

**Q: 如何处理 Agent 之间的冲突？**
- A:
  1. **裁决机制**：引入第三方 Agent 判断
  2. **投票机制**：多数决策
  3. **协商机制**：Agent 之间讨论妥协
  4. **权重分配**：根据可信度加权
  5. **人工介入**：重大冲突由人工决定

**Q: LangGraph 在多 Agent 中的作用？**
- A:
  1. **状态管理**：维护共享状态
  2. **流程控制**：定义 Agent 执行顺序
  3. **条件路由**：根据结果决定下一步
  4. **可视化**：图形化展示工作流
  5. **持久化**：支持长时间运行的任务

**Q: 如何优化多 Agent 的性能？**
- A:
  1. **并行执行**：独立的 Agent 同时运行
  2. **缓存结果**：相同输入直接返回缓存
  3. **精简通信**：减少 Agent 间的数据传递
  4. **超时控制**：防止单个 Agent 阻塞
  5. **动态路由**：跳过不必要的 Agent

---

## 总结

本课程学习了多 Agent 协作的核心知识：

| 知识点 | 说明 |
|--------|------|
| **协作模式** | 顺序、并行、层级、辩论、投票、路由 |
| **专业 Agent** | 每个专注一个领域 |
| **顺序编排** | 流水线式执行 |
| **并行编排** | 同时工作提升效率 |
| **路由模式** | 根据任务类型分发 |
| **辩论模式** | 多方案比较选最优 |

**下一步**: L19 将学习 RAG 系统的第一部分——文档分割与嵌入！